In [0]:
%sql
-- Cross-system identity: CRM hashes → mdl_users via ALL hash columns. TWO bridges:
--   Bridge 1 (contacts): contacts.name hash → USERNAME (86) / FIRSTNAME (2,809) / LASTNAME (1,150)
--   Bridge 2 (lead): leads_processed.name — the LEAD record's own hash. Reaches clients whose
--   contact hashes never match the platform (mostly different people than Bridge 1: 314 + 342 → 633 union)
--   mdl EMAIL is 100% masked (user_…@masked.com) with ZERO overlap vs CRM masked emails —
--   email matching is impossible on this export; name-hash equality is the only bridge
CREATE OR REPLACE TABLE crm_ingestion.gold.days_since_last_login AS (
WITH current_leads AS (
    -- Latest version per lead FIRST (all leads); current-client status filtered at the END.
    -- Filter order matters: filtering status before the QUALIFY selects leads that were EVER
    -- Coaching Client (1,639); filtering after selects CURRENT clients (1,371), matching lead_dim.
    SELECT id AS lead_id, name AS lead_hash, status_label
    FROM crm_ingestion.silver.leads_processed
    QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC) = 1
),
crm_to_moodle AS (
    SELECT lead_id, LASTACCESS FROM (
        SELECT DISTINCT
            c.leads_processed_id AS lead_id,
            m.LASTACCESS
        FROM crm_ingestion.silver.leads_processed_contacts c
        JOIN crm_ingestion.silver.mdl_users_processed m ON c.name = m.USERNAME
        WHERE m.LASTACCESS IS NOT NULL AND m.LASTACCESS != ''
        UNION
        SELECT DISTINCT
            c.leads_processed_id AS lead_id,
            m.LASTACCESS
        FROM crm_ingestion.silver.leads_processed_contacts c
        JOIN crm_ingestion.silver.mdl_users_processed m ON c.name = m.FIRSTNAME
        WHERE m.LASTACCESS IS NOT NULL AND m.LASTACCESS != ''
        UNION
        SELECT DISTINCT
            c.leads_processed_id AS lead_id,
            m.LASTACCESS
        FROM crm_ingestion.silver.leads_processed_contacts c
        JOIN crm_ingestion.silver.mdl_users_processed m ON c.name = m.LASTNAME
        WHERE m.LASTACCESS IS NOT NULL AND m.LASTACCESS != ''
        UNION
        -- Bridge 2: the LEAD record's own hash
        SELECT DISTINCT
            l.lead_id,
            m.LASTACCESS
        FROM current_leads l
        JOIN crm_ingestion.silver.mdl_users_processed m ON l.lead_hash = m.USERNAME
        WHERE m.LASTACCESS IS NOT NULL AND m.LASTACCESS != ''
        UNION
        SELECT DISTINCT
            l.lead_id,
            m.LASTACCESS
        FROM current_leads l
        JOIN crm_ingestion.silver.mdl_users_processed m ON l.lead_hash = m.FIRSTNAME
        WHERE m.LASTACCESS IS NOT NULL AND m.LASTACCESS != ''
        UNION
        SELECT DISTINCT
            l.lead_id,
            m.LASTACCESS
        FROM current_leads l
        JOIN crm_ingestion.silver.mdl_users_processed m ON l.lead_hash = m.LASTNAME
        WHERE m.LASTACCESS IS NOT NULL AND m.LASTACCESS != ''
    )
    QUALIFY ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY LASTACCESS DESC) = 1
)
SELECT
    cc.lead_id,
    DATEDIFF(CURRENT_DATE, DATE(m.LASTACCESS)) AS days_since_last_login
FROM current_leads cc
JOIN crm_to_moodle m ON cc.lead_id = m.lead_id
WHERE cc.status_label = 'Coaching Client'
)

In [0]:
%sql
-- VALIDATION — coverage and population of the login metric.
-- metric_rows should equal clients_matched; rows_outside_client_population must be 0
-- (former clients and non-clients must never appear in this table).
SELECT
    (SELECT COUNT(*) FROM crm_ingestion.gold.days_since_last_login)                    AS metric_rows,
    (SELECT COUNT(*) FROM crm_ingestion.gold.lead_dim)                                 AS client_population,
    (SELECT COUNT(DISTINCT l.lead_id)
     FROM crm_ingestion.gold.days_since_last_login l
     JOIN crm_ingestion.gold.lead_dim d ON d.lead_id = l.lead_id)                      AS clients_matched,
    (SELECT COUNT(*) FROM crm_ingestion.gold.lead_dim)
  - (SELECT COUNT(DISTINCT l.lead_id)
     FROM crm_ingestion.gold.days_since_last_login l
     JOIN crm_ingestion.gold.lead_dim d ON d.lead_id = l.lead_id)                      AS clients_without_login_evidence,
    (SELECT COUNT(*)
     FROM crm_ingestion.gold.days_since_last_login l
     LEFT JOIN crm_ingestion.gold.lead_dim d ON d.lead_id = l.lead_id
     WHERE d.lead_id IS NULL)                                                          AS rows_outside_client_population,
    (SELECT MIN(days_since_last_login) FROM crm_ingestion.gold.days_since_last_login)  AS min_days,
    (SELECT MAX(days_since_last_login) FROM crm_ingestion.gold.days_since_last_login)  AS max_days,
    -- Moodle defaults LASTACCESS to epoch 0 (1970) for users who never logged in —
    -- those users carry NO login evidence and must be watched here.
    (SELECT COUNT(*) FROM crm_ingestion.gold.days_since_last_login
     WHERE days_since_last_login > 3000)                                                AS epoch_default_rows,
    (SELECT MAX(days_since_last_login) FROM crm_ingestion.gold.days_since_last_login
     WHERE days_since_last_login <= 3000)                                               AS max_days_plausible